In [ ]:
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pydicom
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning)

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

SLOTS = [
    ('SAG_FLUID_FS', 'Sagittal', True, True),
    ('COR_FLUID_FS', 'Coronal', True, True),
    ('AX_FLUID_FS',  'Axial', True, True),
    ('SAG_FLUID_NOFS', 'Sagittal', True, False),
    ('COR_T1', 'Coronal', False, False),
    ('SAG_T1', 'Sagittal', False, False),
]

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bsting\b|\bfatsup\b')
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')


class KneeImageProcessor:
    def __init__(
        self,
        img_size: int = 224,
        crop_mm: float = 160.0,
        n_best_slices: int = 12
    ):
        self.img_size = img_size
        self.crop_mm = crop_mm
        self.n_best_slices = n_best_slices

    def parse_dicom_meta(self, dcm_path: Path) -> Dict[str, Any]:
        try:
            ds = pydicom.dcmread(dcm_path, stop_before_pixels=True, force=True)
            
            series_desc = str(getattr(ds, 'SeriesDescription', '') or '')
            seq_name = str(getattr(ds, 'SequenceName', '') or '')
            scan_opts = str(getattr(ds, 'ScanOptions', '') or '')
            scan_seq = str(getattr(ds, 'ScanningSequence', '') or '')
            
            tr = getattr(ds, 'RepetitionTime', None)
            te = getattr(ds, 'EchoTime', None)
            laterality = str(getattr(ds, 'Laterality', '') or '').strip().upper()
            
            slice_thickness = getattr(ds, 'SliceThickness', None)
            flip_angle = getattr(ds, 'FlipAngle', None)
            mfg_raw = str(getattr(ds, 'Manufacturer', '') or '').upper()
            
            if 'SIEMENS' in mfg_raw:
                manufacturer_code = 1.0
            elif 'GE' in mfg_raw or 'GENERAL ELECTRIC' in mfg_raw:
                manufacturer_code = 0.5
            elif 'PHILIPS' in mfg_raw:
                manufacturer_code = 0.0
            else:
                manufacturer_code = -0.5
            
            sex = str(getattr(ds, 'PatientSex', 'O') or 'O').strip().upper()
            age_raw = str(getattr(ds, 'PatientAge', '') or '')
            field_strength = getattr(ds, 'MagneticFieldStrength', 1.5)
            
            age = 50.0
            if age_raw:
                match = re.match(r'(\d+)([YMWD])', age_raw)
                if match:
                    val, unit = match.groups()
                    val = float(val)
                    if unit == 'Y': age = val
                    elif unit == 'M': age = val / 12.0
                    elif unit == 'W': age = val / 52.0
                    elif unit == 'D': age = val / 365.0
            
            pixel_spacing = getattr(ds, 'PixelSpacing', None)
            px = float(pixel_spacing[0]) if pixel_spacing else 0.45 
            
            return {
                'SeriesDescription': series_desc,
                'SequenceName': seq_name,
                'ScanOptions': scan_opts,
                'ScanningSequence': scan_seq,
                'RepetitionTime': float(tr) if tr is not None else None,
                'EchoTime': float(te) if te is not None else None,
                'Laterality': laterality[0] if laterality and laterality[0] in ('L', 'R') else None,
                'PixelSpacing': px,
                'PatientSex': 1.0 if sex == 'M' else (0.0 if sex == 'F' else 0.5),
                'PatientAge': age,
                'MagneticFieldStrength': float(field_strength) if field_strength is not None else 1.5,
                'SliceThickness': float(slice_thickness) if slice_thickness is not None else None,
                'FlipAngle': float(flip_angle) if flip_angle is not None else None,
                'Manufacturer': manufacturer_code
            }
        except Exception:
            return {}

    def classify_series(self, meta: Dict[str, Any], plane: str) -> Tuple[bool, bool]:
        desc = (meta.get('SeriesDescription', '') + ' ' + meta.get('SequenceName', '')).lower()
        desc = re.sub(_SEP, ' ', desc)
        
        opts_fs = any(t.strip() in FATSAT_OPTS for t in meta.get('ScanOptions', '').upper().split('|'))
        fatsat = bool(re.search(_FATSAT_RX, desc) or opts_fs)
        
        tr = meta.get('RepetitionTime')
        te = meta.get('EchoTime')
        gre = 'GR' in meta.get('ScanningSequence', '').upper()
        
        t1 = bool(re.search(_T1_RX, desc))
        t2 = bool(re.search(_T2_RX, desc))
        pdw = bool(re.search(_PD_RX, desc))
        
        if t1 and not t2 and not pdw:
            weight = 'T1'
        elif t2 and not pdw:
            weight = 'T2'
        elif pdw:
            weight = 'PD'
        elif gre:
            weight = 'GRE'
        elif tr is not None and tr < 800:
            weight = 'T1'
        elif te is not None and te > 60:
            weight = 'T2'
        elif tr is not None and tr >= 800:
            weight = 'PD'
        else:
            weight = 'UNK'
            
        fluid = weight in ['PD', 'T2']
        return fluid, fatsat

    def get_physically_sorted_slices(self, series_dir: Path, plane: str) -> List[Tuple[float, Path]]:
        files = list(series_dir.glob("*.dcm"))
        if not files:
            return []
            
        slice_positions = []
        for f in files:
            try:
                ds = pydicom.dcmread(f, stop_before_pixels=True, force=True)
                ipp = getattr(ds, 'ImagePositionPatient', None)
                if ipp is not None:
                    if plane == 'Sagittal': pos = float(ipp[0])
                    elif plane == 'Coronal': pos = float(ipp[1])
                    else: pos = float(ipp[2])
                else:
                    pos = float(getattr(ds, 'SliceLocation', 0.0))
            except Exception:
                num_match = re.findall(r'\d+', f.name)
                pos = float(num_match[-1]) if num_match else 0.0
            slice_positions.append((pos, f))
            
        slice_positions.sort(key=lambda x: x[0])
        return slice_positions

    def process_volume(
        self, 
        sorted_slices: List[Tuple[float, Path]], 
        plane: str, 
        laterality: Optional[str], 
        pixel_spacing: float
    ) -> np.ndarray:
        n = len(sorted_slices)
        if n == 0:
            return np.zeros((self.n_best_slices, self.img_size, self.img_size), dtype=np.uint8)

        lo, hi = int(0.10 * (n - 1)), int(0.90 * (n - 1))
        if hi > lo:
            idx = np.unique(np.linspace(lo, hi, self.n_best_slices).astype(int))
        else:
            idx = np.array([n // 2])
            
        while len(idx) < self.n_best_slices:
            idx = np.append(idx, idx[-1])
            
        planes = []
        for i in idx[:self.n_best_slices]:
            _, fpath = sorted_slices[int(i)]
            try:
                ds = pydicom.dcmread(fpath, force=True)
                a = ds.pixel_array.astype(np.float32)
                
                sl = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
                ic = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
                a = a * sl + ic
                
                if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
                    a = a.max() - a
            except Exception:
                a = np.zeros((self.img_size, self.img_size), dtype=np.float32)
            planes.append(a)

        shp = planes[0].shape
        planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
        vol = np.stack(planes)

        if pixel_spacing > 0:
            want = int(round(self.crop_mm / pixel_spacing))
            h, w = shp
            if 16 < want < min(h, w):
                cy, cx = h // 2, w // 2
                half = want // 2
                vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

        lo_v, hi_v = np.percentile(vol, [1, 99])
        vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0.0, 1.0)

        t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
        t = F.interpolate(t, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False).squeeze(0)

        return (t * 255.0).round().clamp(0, 255).to(torch.uint8).numpy()

    def _compile_meta_features(self, parsed_series: List[Dict[str, Any]], study_mask: np.ndarray) -> np.ndarray:
        if not parsed_series:
            return np.array([0.5, 0.5, 0.5, 0.6, 0.45, -0.5, 0.25, 0.5, 0.5, 0.0], dtype=np.float32)

        patient_sex = parsed_series[0]['PatientSex']
        patient_age = parsed_series[0]['PatientAge']
        field_strength = parsed_series[0]['MagneticFieldStrength']

        slice_thicknesses = [s['SliceThickness'] for s in parsed_series if s.get('SliceThickness') is not None]
        pixel_spacings = [s['PixelSpacing'] for s in parsed_series if s.get('PixelSpacing') is not None]
        manufacturers = [s['Manufacturer'] for s in parsed_series if s.get('Manufacturer') is not None]
        tes = [s['EchoTime'] for s in parsed_series if s.get('EchoTime') is not None]
        trs = [s['RepetitionTime'] for s in parsed_series if s.get('RepetitionTime') is not None]
        flip_angles = [s['FlipAngle'] for s in parsed_series if s.get('FlipAngle') is not None]

        mean_thickness = np.mean(slice_thicknesses) if slice_thicknesses else 3.0
        mean_spacing = np.mean(pixel_spacings) if pixel_spacings else 0.45
        mfg_code = max(set(manufacturers), key=manufacturers.count) if manufacturers else -0.5
        mean_te = np.mean(tes) if tes else 30.0
        mean_tr = np.mean(trs) if trs else 2000.0
        mean_flip = np.mean(flip_angles) if flip_angles else 90.0

        slot_completion_ratio = float(np.sum(study_mask) / len(study_mask))

        meta_arr = np.array([
            patient_sex,                                         # [0] 性别 (M=1.0, F=0.0, O=0.5)
            np.clip(patient_age / 100.0, 0.0, 1.0),              # [1] 年龄 (归一化到 0-1)
            np.clip(field_strength / 3.0, 0.0, 1.5),             # [2] 场强 (0.5=1.5T, 1.0=3T)
            np.clip(mean_thickness / 5.0, 0.0, 2.0),             # [3] 层厚 (典型值 3-4mm -> 0.6-0.8)
            np.clip(mean_spacing / 1.0, 0.0, 2.0),               # [4] PixelSpacing (典型值 0.45)
            mfg_code,                                            # [5] 扫描仪品牌编码 (-0.5~1.0)
            np.clip(mean_te / 120.0, 0.0, 2.0),                  # [6] EchoTime
            np.clip(mean_tr / 4000.0, 0.0, 2.0),                 # [7] RepetitionTime
            np.clip(mean_flip / 180.0, 0.0, 1.0),                # [8] FlipAngle 翻转角
            slot_completion_ratio                                # [9] 有效插槽比例 (0.0~1.0)
        ], dtype=np.float32)

        return meta_arr

    def process_study(self, study_dir: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        study_data = np.zeros((len(SLOTS), self.n_best_slices, self.img_size, self.img_size), dtype=np.uint8)
        study_mask = np.zeros(len(SLOTS), dtype=np.float32)
        
        series_dirs = [p for p in study_dir.iterdir() if p.is_dir()]
        
        parsed_series = []
        for s_dir in series_dirs:
            dcms = list(s_dir.glob("*.dcm"))
            if not dcms:
                continue
            mid_dcm = dcms[len(dcms) // 2]
            meta = self.parse_dicom_meta(mid_dcm)
            if not meta:
                continue
            
            plane = self._infer_anatomical_plane(mid_dcm)
            fluid, fatsat = self.classify_series(meta, plane)
            
            meta.update({
                'plane': plane,
                'fluid': fluid,
                'fatsat': fatsat,
                'dir': s_dir,
                'n_slices': len(dcms)
            })
            parsed_series.append(meta)
            
        laterality = None
        if parsed_series:
            lats = [s['Laterality'] for s in parsed_series if s['Laterality']]
            if lats:
                laterality = max(set(lats), key=lats.count)

        for k, (slot_name, plane, fluid, fs) in enumerate(SLOTS):
            candidates = [
                s for s in parsed_series 
                if s['plane'] == plane and s['fatsat'] == fs and s['fluid'] == fluid
            ]
            if not candidates and not fluid:
                candidates = [s for s in parsed_series if s['plane'] == plane and not s['fatsat']]
                
            if candidates:
                best_s = sorted(candidates, key=lambda x: x['n_slices'], reverse=True)[0]
                sorted_slices = self.get_physically_sorted_slices(best_s['dir'], plane)
                vol = self.process_volume(sorted_slices, plane, laterality, best_s['PixelSpacing'])
                
                study_data[k] = vol
                study_mask[k] = 1.0

        meta_features = self._compile_meta_features(parsed_series, study_mask)
        
        return study_data, study_mask, meta_features

    def _infer_anatomical_plane(self, dcm_path: Path) -> str:
        try:
            ds = pydicom.dcmread(dcm_path, stop_before_pixels=True, force=True)
            iop = ds.ImageOrientationPatient
            if iop is None or len(iop) < 6:
                return 'Sagittal'
                
            row = np.array(iop[0:3])
            col = np.array(iop[3:6])
            normal = np.cross(row, col)
            
            abs_normal = np.abs(normal)
            max_idx = np.argmax(abs_normal)
            
            if max_idx == 0: return 'Sagittal'
            elif max_idx == 1: return 'Coronal'
            else: return 'Axial'
        except Exception:
            return 'Sagittal'

In [ ]:
import os
import gc
import re
import time
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

IMG_SIZE = 224
CROP_MM = 160.0
N_BEST_SLICES = 12
STUDIES_PER_CHUNK = 400
START_CHUNK = 0

SLOTS = [
    ('SAG_FLUID_FS', 'Sagittal', True, True),
    ('COR_FLUID_FS', 'Coronal', True, True),
    ('AX_FLUID_FS',  'Axial', True, True),
    ('SAG_FLUID_NOFS', 'Sagittal', True, False),
    ('COR_T1', 'Coronal', False, False),
    ('SAG_T1', 'Sagittal', False, False),
]

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bsting\b|\bfatsup\b')
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')

def find_root():
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection')]:
        if (c / 'train.csv').is_file(): return c
    raise FileNotFoundError('RSNA Knee Dataset not found!')

ROOT = find_root()
OUTPUT_DIR = Path('/kaggle/working/rsna_knee_processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_WORKERS = os.cpu_count() or 4

print(f"[Init] Input root: {ROOT}")
print(f"[Init] Output directory: {OUTPUT_DIR}")
print(f"[Init] CPU cores available: {N_WORKERS}")

def parse_age(age_str: str) -> float:
    if not age_str:
        return 50.0
    match = re.match(r'(\d+)([YMWD])', age_str.strip().upper())
    if match:
        val, unit = match.groups()
        val = float(val)
        if unit == 'Y': return val
        elif unit == 'M': return val / 12.0
        elif unit == 'W': return val / 52.0
        elif unit == 'D': return val / 365.0
    return 50.0

def parse_single_series_meta(item) -> dict:
    study_uid, series_uid, path = item
    row = {
        'StudyInstanceUID': study_uid, 
        'SeriesInstanceUID': series_uid, 
        'dir': str(path),
        'PixelSpacing': 0.45,
        'PatientSex': 0.5,
        'PatientAge': 50.0,
        'MagneticFieldStrength': 1.5,
        'Laterality': None,
        'SliceThickness': None,
        'FlipAngle': None,
        'Manufacturer': -0.5
    }
    try:
        files = sorted([e.name for e in os.scandir(path) if e.name.endswith('.dcm')])
        row['n_slices'] = len(files)
        row['files'] = files
        if files:
            mid_file = os.path.join(path, files[len(files) // 2])
            ds = pydicom.dcmread(mid_file, stop_before_pixels=True, force=True)
            
            row['SeriesDescription'] = str(getattr(ds, 'SeriesDescription', '') or '')
            row['SequenceName'] = str(getattr(ds, 'SequenceName', '') or '')
            row['ScanOptions'] = str(getattr(ds, 'ScanOptions', '') or '')
            row['ScanningSequence'] = str(getattr(ds, 'ScanningSequence', '') or '')
            row['RepetitionTime'] = getattr(ds, 'RepetitionTime', None)
            row['EchoTime'] = getattr(ds, 'EchoTime', None)
            
            lat = str(getattr(ds, 'Laterality', '') or '').strip().upper()
            if lat and lat[0] in ('L', 'R'):
                row['Laterality'] = lat[0]
                
            sex = str(getattr(ds, 'PatientSex', 'O') or 'O').strip().upper()
            row['PatientSex'] = 1.0 if sex == 'M' else (0.0 if sex == 'F' else 0.5)
            row['PatientAge'] = parse_age(str(getattr(ds, 'PatientAge', '')))
            row['MagneticFieldStrength'] = float(getattr(ds, 'MagneticFieldStrength', 1.5) or 1.5)
            
            row['SliceThickness'] = getattr(ds, 'SliceThickness', None)
            row['FlipAngle'] = getattr(ds, 'FlipAngle', None)
            
            mfg = str(getattr(ds, 'Manufacturer', '') or '').upper()
            if 'SIEMENS' in mfg: row['Manufacturer'] = 1.0
            elif 'GE' in mfg or 'GENERAL ELECTRIC' in mfg: row['Manufacturer'] = 0.5
            elif 'PHILIPS' in mfg: row['Manufacturer'] = 0.0
            else: row['Manufacturer'] = -0.5
            
            px_val = getattr(ds, 'PixelSpacing', None)
            if px_val:
                row['PixelSpacing'] = float(px_val[0])
    except Exception as e:
        row['err'] = str(e)[:50]
        row['n_slices'] = 0
    return row

def annotate_series(df: pd.DataFrame) -> pd.DataFrame:
    desc = (df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')).str.lower().str.replace(_SEP, ' ', regex=True)
    opts_fs = df['ScanOptions'].fillna('').str.upper().str.split('|').apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1',
                    np.where(t2 & ~pdw, 'T2',
                      np.where(pdw, 'PD',
                        np.where(gre, 'GRE',
                          np.where(tr < 800, 'T1',
                            np.where(te > 60, 'T2',
                              np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    return df

def get_physically_sorted_slices(series_dir: Path, files: list, plane: str) -> list:
    slice_positions = []
    for f in files:
        fpath = series_dir / f
        try:
            ds = pydicom.dcmread(fpath, stop_before_pixels=True, force=True)
            ipp = getattr(ds, 'ImagePositionPatient', None)
            if ipp is not None:
                if plane == 'Sagittal': pos = float(ipp[0])
                elif plane == 'Coronal': pos = float(ipp[1])
                else: pos = float(ipp[2])
            else:
                pos = float(getattr(ds, 'SliceLocation', 0.0))
        except Exception:
            num_match = re.findall(r'\d+', f)
            pos = float(num_match[-1]) if num_match else 0.0
        slice_positions.append((pos, fpath))
    slice_positions.sort(key=lambda x: x[0])
    return slice_positions

def process_slot_volume(rec: dict, plane: str, lat: str) -> np.ndarray:
    if not rec or rec.get('n_slices', 0) == 0:
        return None

    files = rec['files']
    s_dir = Path(rec['dir'])
    n = len(files)
    px = rec['PixelSpacing']

    lo, hi = int(0.10 * (n - 1)), int(0.90 * (n - 1))
    if hi > lo:
        idx = np.unique(np.linspace(lo, hi, N_BEST_SLICES).astype(int))
    else:
        idx = np.array([n // 2])
    while len(idx) < N_BEST_SLICES:
        idx = np.append(idx, idx[-1])

    sorted_slices = get_physically_sorted_slices(s_dir, files, plane)
    
    planes = []
    for i in idx[:N_BEST_SLICES]:
        try:
            _, fpath = sorted_slices[int(i)]
            ds = pydicom.dcmread(fpath, force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
            if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
                a = a.max() - a
        except Exception:
            a = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        planes.append(a)

    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    if px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0.0, 1.0)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze(0)

    return (t * 255.0).round().clamp(0, 255).to(torch.uint8).numpy()

def process_single_study(args) -> tuple:
    st_uid, study_slots, lat_val, meta_feats = args
    study_data = np.zeros((len(SLOTS), N_BEST_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    study_mask = np.zeros(len(SLOTS), dtype=np.float32)

    for k, (slot_name, plane, _, _) in enumerate(SLOTS):
        if slot_name in study_slots:
            vol = process_slot_volume(study_slots[slot_name], plane, lat_val)
            if vol is not None:
                study_data[k] = vol
                study_mask[k] = 1.0

    return st_uid, study_data, study_mask, meta_feats

def main():
    t0 = time.time()
    
    print("\n=== [Step 1] Scanning DICOM structure ===")
    train_series_csv = pd.read_csv(ROOT / 'train_series.csv')
    plane_map = dict(zip(train_series_csv['SeriesInstanceUID'], train_series_csv['Anatomical_Plane']))

    items = []
    base_dir = ROOT / 'train_series'
    for study in os.scandir(base_dir):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((study.name, series.name, series.path))

    print("\n=== [Step 2] Reading raw DICOM headers ===")
    with ThreadPoolExecutor(max_workers=N_WORKERS * 2) as pool:
        rows = list(tqdm(pool.map(parse_single_series_meta, items), total=len(items), desc='Parsing Headers'))

    df = pd.DataFrame(rows)
    df = df[df['n_slices'] > 0].reset_index(drop=True)
    df['plane'] = df['SeriesInstanceUID'].map(plane_map)
    df = annotate_series(df)

    lat_map = {}
    for st, g in df.groupby('StudyInstanceUID'):
        v = [x for x in g['Laterality'].dropna()]
        lat_map[st] = max(set(v), key=v.count) if v else None

    print("\n=== [Step 3] Matching series to clinic slots ===")
    matched_slots = {}
    study_metas = {}
    
    temp_processor = KneeImageProcessor(img_size=IMG_SIZE, crop_mm=CROP_MM, n_best_slices=N_BEST_SLICES)

    for study, g in tqdm(df.groupby('StudyInstanceUID'), desc="Matching Slots"):
        chosen = {}
        parsed_list = []
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs) & (g['fluid'] == fluid)
            cand = g[sel]
            if len(cand) == 0 and not fluid:
                cand = g[(g['plane'] == plane) & (~g['fatsat'])]
            if len(cand) > 0:
                rec_dict = cand.sort_values('n_slices', ascending=False).iloc[0].to_dict()
                chosen[name] = rec_dict
                parsed_list.append(rec_dict)
        matched_slots[study] = chosen
        
        study_mask = np.zeros(len(SLOTS), dtype=np.float32)
        for k, (slot_name, _, _, _) in enumerate(SLOTS):
            if slot_name in chosen:
                study_mask[k] = 1.0

        study_metas[study] = temp_processor._compile_meta_features(parsed_list, study_mask)

    print("\n=== [Step 4] Preprocessing & Saving Chunks ===")
    studies = sorted(matched_slots.keys())
    num_chunks = int(np.ceil(len(studies) / STUDIES_PER_CHUNK))
    manifest = []

    for chunk_idx in range(num_chunks):
        c_start = chunk_idx * STUDIES_PER_CHUNK
        c_end = min((chunk_idx + 1) * STUDIES_PER_CHUNK, len(studies))
        chunk_studies = studies[c_start:c_end]
        chunk_file_name = f'train_chunk_{chunk_idx:02d}.npz'

        if chunk_idx < START_CHUNK:
            print(f'[{time.time()-t0:.1f}s] [Skipped] Chunk {chunk_idx:02d} ({chunk_file_name})')
            for uid in chunk_studies:
                manifest.append({'StudyInstanceUID': uid, 'chunk_file': chunk_file_name})
            continue

        job_args = [
            (uid, matched_slots[uid], lat_map.get(uid), study_metas[uid]) 
            for uid in chunk_studies
        ]
        
        chunk_uids, chunk_imgs, chunk_masks, chunk_metas = [], [], [], []

        print(f'\nProcessing Chunk {chunk_idx:02d}/{num_chunks-1} (Studies {c_start} ~ {c_end - 1})...')
        with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
            results = list(tqdm(pool.map(process_single_study, job_args), total=len(job_args), desc=f'Chunk {chunk_idx:02d}'))

        for uid, imgs, mask, meta in results:
            chunk_uids.append(uid)
            chunk_imgs.append(imgs)
            chunk_masks.append(mask)
            chunk_metas.append(meta)
            manifest.append({'StudyInstanceUID': uid, 'chunk_file': chunk_file_name})

        chunk_file_path = OUTPUT_DIR / chunk_file_name
        np.savez_compressed(
            chunk_file_path,
            uids=np.array(chunk_uids),
            images=np.stack(chunk_imgs),
            masks=np.stack(chunk_masks),
            metas=np.stack(chunk_metas)
        )
        size_mb = chunk_file_path.stat().st_size / (1024 * 1024)
        print(f'[{time.time()-t0:.1f}s] Saved: {chunk_file_name} ({size_mb:.1f} MB)')
        
        gc.collect()

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(OUTPUT_DIR / 'manifest.csv', index=False)
    print(f'\n=== Success ===\nManifest saved to manifest.csv. Total studies: {len(manifest_df)}')
    print(f'Total time elapsed: {time.time()-t0:.1f} seconds.')

if __name__ == '__main__':
    main()